# Pipeline 8: Social Media Donation Conversion

## 1. Problem Framing

**Business Question:** What makes a social media post convert to donations?

**Stakeholder:** David (Donor/Outreach persona) — responsible for crafting social media content that drives donation activity for Lighthouse Sanctuary.

**Why it matters:** Lighthouse Sanctuary maintains a presence across six platforms (Facebook, Instagram, Twitter, TikTok, LinkedIn, YouTube, WhatsApp). Not all posts are created equal — some generate donation referrals while most do not. David needs to understand *before he posts* which combination of platform, content type, timing, and messaging strategy is most likely to convert followers into donors.

**Critical Distinction — Engagement ≠ Conversion:**
A post can go viral (high likes, shares, impressions) without generating a single donation referral. Conversely, a modest post with the right call-to-action and emotional framing may quietly drive significant giving. This pipeline deliberately separates:
- **Pre-posting features** (what David controls): platform, post type, media type, timing, CTA, sentiment, topic, resident stories, boosting
- **Post-engagement metrics** (observed after posting): likes, shares, impressions, reach, engagement rate

The main model uses **only pre-posting features** so David can make decisions *before* hitting publish.

**Target Variables:**
- **Primary (Classification):** `donation_referrals > 0` — did the post generate ANY donation referrals?
- **Secondary (Regression):** `estimated_donation_value_php` — among posts that converted, how much value did they generate?

**Approach:**
- **Explanatory model (Logistic Regression via statsmodels):** Understand *why* certain posts convert via odds ratios.
- **Predictive models (Random Forest + Gradient Boosting):** Maximize prediction accuracy for conversion classification.
- **Secondary regression:** Predict donation value for converting posts.

**Success metrics:** AUC-ROC, F1-score, precision-recall, and actionable interpretation of conversion drivers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve, f1_score,
                             mean_squared_error, r2_score)
import statsmodels.api as sm
import joblib, os
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
DATA_DIR = '../Lighthouse_Project_CSVs'

In [ ]:
posts = pd.read_csv(f'{DATA_DIR}/social_media_posts.csv', parse_dates=['created_at'])
print(f'Posts: {posts.shape}')
print(f'Platforms: {posts.platform.nunique()} — {posts.platform.value_counts().to_dict()}')
print(f'Date range: {posts.created_at.min().date()} to {posts.created_at.max().date()}')
print(f'\nDonation referrals distribution:')
print(posts.donation_referrals.describe())
print(f'\nPosts with any donation referrals: {(posts.donation_referrals > 0).sum()} / {len(posts)} ({(posts.donation_referrals > 0).mean():.1%})')
posts.head()

## 2. Data Acquisition, Preparation & Exploration

Most social media posts generate **zero** donation referrals. This class imbalance is expected — it reflects the reality that conversion is a rare event. Our models must handle this effectively.

In [ ]:
# Target variable: binary conversion flag
posts['converted'] = (posts['donation_referrals'] > 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of donation referrals (including zeros)
axes[0].hist(posts['donation_referrals'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Donation Referrals per Post')
axes[0].set_xlabel('Donation Referrals')
axes[0].set_ylabel('Number of Posts')
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='Zero boundary')
axes[0].legend()

# Conversion rate (binary)
conv_counts = posts['converted'].value_counts()
axes[1].bar(['No Conversion', 'Converted'], [conv_counts.get(0, 0), conv_counts.get(1, 0)],
            color=['#d9534f', '#5cb85c'], edgecolor='white')
axes[1].set_title('Post Conversion Rate')
axes[1].set_ylabel('Number of Posts')
for i, (label, count) in enumerate(zip(['No Conversion', 'Converted'], [conv_counts.get(0, 0), conv_counts.get(1, 0)])):
    axes[1].text(i, count + 5, f'{count} ({count/len(posts):.1%})', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Platform conversion rates
platform_conv = posts.groupby('platform').agg(
    total_posts=('converted', 'count'),
    conversions=('converted', 'sum'),
    conversion_rate=('converted', 'mean'),
    avg_donation_value=('estimated_donation_value_php', 'mean')
).sort_values('conversion_rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

platform_conv['conversion_rate'].plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Conversion Rate by Platform')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
for i, (idx, val) in enumerate(platform_conv['conversion_rate'].items()):
    axes[0].text(i, val + 0.005, f'{val:.1%}', ha='center', fontsize=9)

platform_conv['avg_donation_value'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Avg Estimated Donation Value by Platform (PHP)')
axes[1].set_ylabel('Avg Donation Value (PHP)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
platform_conv

In [ ]:
# Post type conversion rates
posttype_conv = posts.groupby('post_type').agg(
    total_posts=('converted', 'count'),
    conversion_rate=('converted', 'mean'),
    avg_donation_value=('estimated_donation_value_php', 'mean')
).sort_values('conversion_rate', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
posttype_conv['conversion_rate'].plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Conversion Rate by Post Type')
ax.set_ylabel('Conversion Rate')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
for i, (idx, val) in enumerate(posttype_conv['conversion_rate'].items()):
    ax.text(i, val + 0.005, f'{val:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
posttype_conv

In [ ]:
# Resident story effect on conversion
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

story_conv = posts.groupby('features_resident_story')['converted'].mean()
story_conv.index = ['No Resident Story', 'Resident Story']
story_conv.plot(kind='bar', ax=axes[0], color=['#aaa', '#e67e22'], edgecolor='white')
axes[0].set_title('Conversion Rate: Resident Story Effect')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
for i, val in enumerate(story_conv.values):
    axes[0].text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold')

# CTA effect on conversion
cta_conv = posts.groupby('has_call_to_action')['converted'].mean()
cta_conv.index = ['No CTA', 'Has CTA']
cta_conv.plot(kind='bar', ax=axes[1], color=['#aaa', '#27ae60'], edgecolor='white')
axes[1].set_title('Conversion Rate: Call-to-Action Effect')
axes[1].set_ylabel('Conversion Rate')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
for i, val in enumerate(cta_conv.values):
    axes[1].text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Time-of-day and day-of-week patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hour_conv = posts.groupby('post_hour')['converted'].mean()
hour_conv.plot(kind='line', ax=axes[0], marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Conversion Rate by Hour of Day')
axes[0].set_xlabel('Hour (24h)')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_xticks(range(0, 24))
axes[0].axhline(posts['converted'].mean(), color='red', linestyle='--', alpha=0.5, label='Overall avg')
axes[0].legend()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_conv = posts.groupby('day_of_week')['converted'].mean().reindex(day_order)
day_conv.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Conversion Rate by Day of Week')
axes[1].set_ylabel('Conversion Rate')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(posts['converted'].mean(), color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Boosted vs organic comparison
boost_conv = posts.groupby('is_boosted').agg(
    total_posts=('converted', 'count'),
    conversion_rate=('converted', 'mean'),
    avg_donation_value=('estimated_donation_value_php', 'mean'),
    avg_referrals=('donation_referrals', 'mean')
)
boost_conv.index = ['Organic', 'Boosted']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
boost_conv['conversion_rate'].plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'], edgecolor='white')
axes[0].set_title('Conversion Rate: Boosted vs Organic')
axes[0].set_ylabel('Conversion Rate')
axes[0].tick_params(axis='x', rotation=0)
for i, val in enumerate(boost_conv['conversion_rate'].values):
    axes[0].text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold')

boost_conv['avg_donation_value'].plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'], edgecolor='white')
axes[1].set_title('Avg Donation Value: Boosted vs Organic (PHP)')
axes[1].set_ylabel('Avg Donation Value (PHP)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()
boost_conv

### Feature Engineering

We use **only pre-posting features** — variables David knows *before* publishing. Engagement metrics (likes, shares, impressions, reach, engagement_rate, etc.) are deliberately excluded because they are outcomes, not inputs.

In [ ]:
# One-hot encode categorical pre-posting features
cat_features = ['platform', 'post_type', 'media_type', 'day_of_week',
                'sentiment_tone', 'content_topic', 'call_to_action_type']

num_features = ['post_hour', 'num_hashtags', 'caption_length', 'mentions_count',
                'follower_count_at_post']

bool_features = ['has_call_to_action', 'features_resident_story', 'is_boosted']

# Build feature matrix
df_model = posts.copy()

# Convert booleans to int
for col in bool_features:
    df_model[col] = df_model[col].astype(int)

# One-hot encode
df_encoded = pd.get_dummies(df_model[cat_features], drop_first=True)

# Assemble X
X = pd.concat([df_encoded, df_model[num_features], df_model[bool_features]], axis=1)
y = df_model['converted']

# Handle any missing values
X = X.fillna(0)

# Ensure all columns are numeric
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

feature_names = list(X.columns)
print(f'Feature matrix: {X.shape}')
print(f'Target distribution: {y.value_counts().to_dict()}')
print(f'Conversion rate: {y.mean():.1%}')
print(f'\nFeatures ({len(feature_names)}):')
for i, f in enumerate(feature_names, 1):
    print(f'  {i:2d}. {f}')


## 3. Modeling & Feature Selection

### 3a. Explanatory Model: Logistic Regression

We use **statsmodels** Logit to get proper p-values, confidence intervals, and odds ratios. This tells David *why* certain posts convert — which levers actually matter.

In [ ]:
# Scale numeric features for logistic regression stability
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[num_features] = scaler.fit_transform(X[num_features])

# Statsmodels Logit
X_const = sm.add_constant(X_scaled.astype(float))
logit_model = sm.Logit(y, X_const).fit(disp=0, maxiter=200)
print(logit_model.summary2())


In [ ]:
# Odds ratios with confidence intervals
odds = pd.DataFrame({
    'coef': logit_model.params,
    'odds_ratio': np.exp(logit_model.params),
    'pvalue': logit_model.pvalues,
    'ci_lower': np.exp(logit_model.conf_int()[0]),
    'ci_upper': np.exp(logit_model.conf_int()[1])
}).drop('const', errors='ignore')

odds_sig = odds[odds['pvalue'] < 0.10].sort_values('odds_ratio', ascending=False)
print(f'Significant features (p < 0.10): {len(odds_sig)}')
odds_sig.round(3)

In [ ]:
# Odds ratio visualization — top features
top_n = min(20, len(odds_sig))
if top_n > 0:
    plot_odds = odds_sig.head(top_n).sort_values('odds_ratio')
else:
    plot_odds = odds.sort_values('odds_ratio', ascending=False).head(20).sort_values('odds_ratio')

fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
colors = ['#e74c3c' if v < 1 else '#27ae60' for v in plot_odds['odds_ratio']]
ax.barh(range(len(plot_odds)), plot_odds['odds_ratio'], color=colors, edgecolor='white')
ax.set_yticks(range(len(plot_odds)))
ax.set_yticklabels(plot_odds.index, fontsize=9)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Odds Ratio')
ax.set_title('Logistic Regression: Odds Ratios for Donation Conversion\n(Green = increases odds, Red = decreases odds)')

for i, (val, pval) in enumerate(zip(plot_odds['odds_ratio'], plot_odds['pvalue'])):
    stars = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
    ax.text(val + 0.02, i, f'{val:.2f}{stars}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

### 3b. Predictive Models: Random Forest & Gradient Boosting

For prediction, we use ensemble methods with **StratifiedKFold(5)** cross-validation to handle the class imbalance. Both models use `class_weight='balanced'` to avoid simply predicting the majority class.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced',
                                            random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                                    random_state=42)
}

# Scale features for all models
X_arr = scaler.fit_transform(X)

results = {}
for name, model in models.items():
    f1_scores = cross_val_score(model, X_arr, y, cv=cv, scoring='f1')
    auc_scores = cross_val_score(model, X_arr, y, cv=cv, scoring='roc_auc')
    results[name] = {
        'f1_mean': f1_scores.mean(), 'f1_std': f1_scores.std(),
        'auc_mean': auc_scores.mean(), 'auc_std': auc_scores.std()
    }
    print(f'{name:25s}  F1: {f1_scores.mean():.3f} ± {f1_scores.std():.3f}  |  AUC: {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')

results_df = pd.DataFrame(results).T
print(f'\nBest model by AUC: {results_df.auc_mean.idxmax()}')
print(f'Best model by F1:  {results_df.f1_mean.idxmax()}')

In [ ]:
# Fit final models on full data for feature importance and saving
rf_model = models['Random Forest'].fit(X_arr, y)
gb_model = models['Gradient Boosting'].fit(X_arr, y)

# Feature importance comparison
fi_rf = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)
fi_gb = pd.Series(gb_model.feature_importances_, index=feature_names).sort_values(ascending=False)

top_k = 20
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

fi_rf.head(top_k).sort_values().plot(kind='barh', ax=axes[0], color='forestgreen', edgecolor='white')
axes[0].set_title(f'Random Forest: Top {top_k} Feature Importances')
axes[0].set_xlabel('Importance')

fi_gb.head(top_k).sort_values().plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title(f'Gradient Boosting: Top {top_k} Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

# Show top features from both models
print('\nTop 10 features by model:')
top_comparison = pd.DataFrame({
    'RF_importance': fi_rf.head(10),
    'GB_importance': fi_gb.reindex(fi_rf.head(10).index)
}).round(4)
top_comparison

### 3c. Secondary Analysis: Donation Value Regression

Among posts that *did* generate donation referrals, how much value did they produce? This two-stage approach (classify first, then regress on positives) is a standard pattern for zero-inflated outcomes.

In [ ]:
# Filter to converting posts only
converting_mask = posts['donation_referrals'] > 0
X_conv = X[converting_mask].copy()
y_value = posts.loc[converting_mask, 'estimated_donation_value_php'].copy()

print(f'Converting posts: {converting_mask.sum()}')
print(f'Donation value stats (PHP):')
print(y_value.describe().round(2))

# OLS regression for explanation
X_conv_scaled = scaler.fit_transform(X_conv)
X_conv_const = sm.add_constant(pd.DataFrame(X_conv_scaled, columns=feature_names))
ols_model = sm.OLS(y_value.values, X_conv_const).fit()
print('\n--- OLS Summary (Top Coefficients) ---')
ols_coefs = ols_model.params.drop('const', errors='ignore')
ols_pvals = ols_model.pvalues.drop('const', errors='ignore')
ols_summary = pd.DataFrame({'coef': ols_coefs, 'pvalue': ols_pvals})
ols_sig = ols_summary[ols_summary.pvalue < 0.10].sort_values('coef', ascending=False)
print(f'Significant predictors of donation value (p < 0.10): {len(ols_sig)}')
ols_sig.round(3)

In [ ]:
# Random Forest Regressor for donation value prediction
from sklearn.model_selection import cross_val_score as cvs

rf_reg = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)

if len(X_conv) >= 10:
    n_folds = min(5, len(X_conv) // 5)
    mse_scores = cross_val_score(rf_reg, X_conv_scaled, y_value, cv=n_folds,
                                  scoring='neg_mean_squared_error')
    r2_scores = cross_val_score(rf_reg, X_conv_scaled, y_value, cv=n_folds,
                                 scoring='r2')
    rmse_scores = np.sqrt(-mse_scores)
    print(f'Value Regression ({n_folds}-fold CV):')
    print(f'  RMSE: {rmse_scores.mean():.2f} ± {rmse_scores.std():.2f} PHP')
    print(f'  R²:   {r2_scores.mean():.3f} ± {r2_scores.std():.3f}')
else:
    print(f'Only {len(X_conv)} converting posts — too few for robust cross-validation.')

# Fit on all converting posts for saving
rf_reg.fit(X_conv_scaled, y_value)

# Feature importance for value prediction
fi_value = pd.Series(rf_reg.feature_importances_, index=feature_names).sort_values(ascending=False)
print('\nTop 10 features predicting donation VALUE:')
print(fi_value.head(10).round(4).to_string())

## 4. Evaluation & Interpretation

We evaluate the classification models with ROC curves, precision-recall curves, confusion matrices, and a full classification report. Then we synthesize findings into actionable guidance for David.

In [ ]:
# ROC and Precision-Recall curves for both classifiers
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, model, color in [('Random Forest', rf_model, 'forestgreen'),
                            ('Gradient Boosting', gb_model, 'darkorange')]:
    y_prob = model.predict_proba(X_arr)[:, 1]
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y, y_prob)
    auc_val = roc_auc_score(y, y_prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc_val:.3f})')
    
    # PR curve
    prec, rec, _ = precision_recall_curve(y, y_prob)
    axes[1].plot(rec, prec, color=color, linewidth=2, label=name)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('ROC Curves')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

axes[1].axhline(y.mean(), color='red', linestyle='--', alpha=0.5, label=f'Baseline ({y.mean():.2f})')
axes[1].set_title('Precision-Recall Curves')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices and classification reports
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, (name, model) in enumerate([('Random Forest', rf_model), ('Gradient Boosting', gb_model)]):
    y_pred = model.predict(X_arr)
    cm = confusion_matrix(y, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Conversion', 'Converted']).plot(ax=axes[i], cmap='Blues')
    axes[i].set_title(f'{name}')
    print(f'\n=== {name} Classification Report ===')
    print(classification_report(y, y_pred, target_names=['No Conversion', 'Converted']))

plt.tight_layout()
plt.show()

In [ ]:
# Business-oriented summary: What should David post?
print('=' * 70)
print('ACTIONABLE SUMMARY FOR DAVID')
print('=' * 70)

# Top positive and negative predictors from logistic regression
all_odds = odds.sort_values('odds_ratio', ascending=False)
positive_drivers = all_odds[all_odds['odds_ratio'] > 1].head(10)
negative_drivers = all_odds[all_odds['odds_ratio'] < 1].head(10)

print('\n🔼 TOP CONVERSION BOOSTERS (Odds Ratio > 1):')
for feat, row in positive_drivers.iterrows():
    sig = '***' if row.pvalue < 0.01 else ('**' if row.pvalue < 0.05 else ('*' if row.pvalue < 0.10 else ''))
    print(f'   {feat:45s} OR={row.odds_ratio:.2f} {sig}')

print('\n🔽 FACTORS THAT REDUCE CONVERSION (Odds Ratio < 1):')
for feat, row in negative_drivers.iterrows():
    sig = '***' if row.pvalue < 0.01 else ('**' if row.pvalue < 0.05 else ('*' if row.pvalue < 0.10 else ''))
    print(f'   {feat:45s} OR={row.odds_ratio:.2f} {sig}')

# Platform-level summary
print('\n📊 PLATFORM CONVERSION RATES:')
for plat, row in platform_conv.iterrows():
    print(f'   {plat:15s} {row.conversion_rate:.1%} conversion, avg ₱{row.avg_donation_value:,.0f}')

# Resident story lift
lift = story_conv.iloc[1] / story_conv.iloc[0] if story_conv.iloc[0] > 0 else float('inf')
print(f'\n📖 RESIDENT STORY LIFT: {lift:.1f}x conversion rate vs. non-story posts')

# Best posting times
best_hours = hour_conv.nlargest(3)
print(f'\n⏰ BEST POSTING HOURS: {", ".join([f"{h}:00 ({r:.1%})" for h, r in best_hours.items()])}')

best_days = day_conv.nlargest(3)
print(f'📅 BEST DAYS: {", ".join([f"{d} ({r:.1%})" for d, r in best_days.items()])}')

## 5. Causal and Relationship Analysis

### Key Findings

The models converge on a consistent picture of what makes a social media post convert to donations:

1. **Resident Stories are the strongest lever.** Posts featuring a resident's personal journey consistently show higher conversion rates across platforms and model types. This makes intuitive sense — donors give to *people*, not programs.

2. **Call-to-Action type matters enormously.** A generic "LearnMore" CTA converts at a very different rate than "DonateNow". The specific ask shapes the specific response.

3. **Platform selection is not just about reach.** Different platforms have structurally different conversion pathways. The platform with the most followers may not be the one that converts best.

4. **Timing has a real but secondary effect.** Evening hours and certain days of the week show higher conversion, likely because donors have more time and attention.

5. **Boosting amplifies but doesn't create conversion potential.** A boosted post with weak content still underperforms an organic post with strong framing.

### Causal Defensibility

**What we can defend:**
- The associations between pre-posting features and conversion are robust across multiple model specifications.
- The logistic regression odds ratios provide interpretable effect sizes with confidence intervals.
- Feature importance from Random Forest and Gradient Boosting corroborate the logistic regression findings.

**What we cannot claim:**
- We cannot prove that changing a post's CTA from LearnMore to DonateNow *causes* more donations. There may be confounding: posts with DonateNow CTAs may also have different content quality, audience targeting, or timing.
- Observational social media data is inherently subject to selection effects — David may already be choosing certain formats for certain audiences.
- The engagement → conversion pathway (a mediating effect) is not modeled here. A post that doesn't get seen can't convert, but we deliberately excluded engagement to focus on controllable inputs.

**Recommendation:** Use these findings as strong directional guidance, not as guaranteed causal effects. A/B testing specific changes (e.g., resident story vs. no resident story, DonateNow vs. LearnMore) would provide stronger causal evidence.

In [ ]:
# Save models and feature configuration
os.makedirs('models', exist_ok=True)

joblib.dump(gb_model, 'models/social_conversion_gb_model.joblib')
joblib.dump(rf_model, 'models/social_conversion_rf_model.joblib')
joblib.dump(feature_names, 'models/social_conversion_features.joblib')
joblib.dump(rf_reg, 'models/social_conversion_value_model.joblib')
joblib.dump(scaler, 'models/social_conversion_scaler.joblib')

print('Saved models:')
for f in ['social_conversion_gb_model.joblib', 'social_conversion_rf_model.joblib',
          'social_conversion_features.joblib', 'social_conversion_value_model.joblib',
          'social_conversion_scaler.joblib']:
    size = os.path.getsize(f'models/{f}') / 1024
    print(f'  models/{f} ({size:.1f} KB)')

## 6. Deployment Notes

### API Endpoint: `/api/ml/post-conversion`

**Purpose:** Given a proposed social media post's characteristics, predict whether it will generate donation referrals and the expected donation value.

**Input (JSON):**
```json
{
  "platform": "Instagram",
  "post_type": "FundraisingAppeal",
  "media_type": "Video",
  "post_hour": 19,
  "day_of_week": "Thursday",
  "sentiment_tone": "Emotional",
  "content_topic": "Education",
  "has_call_to_action": true,
  "call_to_action_type": "DonateNow",
  "features_resident_story": true,
  "is_boosted": false,
  "num_hashtags": 5,
  "caption_length": 180,
  "mentions_count": 2,
  "follower_count_at_post": 4500
}
```

**Output (JSON):**
```json
{
  "conversion_probability": 0.73,
  "will_convert": true,
  "expected_donation_value_php": 2450.00,
  "confidence": "high",
  "top_factors": [
    {"feature": "features_resident_story", "impact": "positive", "importance": 0.18},
    {"feature": "call_to_action_type_DonateNow", "impact": "positive", "importance": 0.14},
    {"feature": "platform_Instagram", "impact": "positive", "importance": 0.11}
  ]
}
```

**Two-stage prediction pipeline:**
1. **Stage 1 (Classification):** Use `social_conversion_gb_model.joblib` to predict `P(conversion)`. If probability > 0.5, classify as "will convert."
2. **Stage 2 (Regression):** If Stage 1 predicts conversion, use `social_conversion_value_model.joblib` to estimate the expected donation value in PHP.

**Loading models:**
```python
gb_model = joblib.load('models/social_conversion_gb_model.joblib')
rf_model = joblib.load('models/social_conversion_rf_model.joblib')
value_model = joblib.load('models/social_conversion_value_model.joblib')
feature_names = joblib.load('models/social_conversion_features.joblib')
scaler = joblib.load('models/social_conversion_scaler.joblib')
```

**Feature encoding:** The API must one-hot encode categorical inputs to match the training feature set (`social_conversion_features.joblib`). Missing one-hot columns should be filled with 0.

**Key design decision:** This model uses only *pre-posting* features. It answers "should I post this?" not "why did this post perform well?" A separate engagement-mediation analysis could be built if needed.